In [3]:
import os
import numpy as np

hurr_path = 'region_hurr.npy'
env_path  = 'region_env.npy'

if os.path.exists(hurr_path) and os.path.exists(env_path):
    print("Skip!")
else:
    hurr_loc = np.load('/global/cfs/cdirs/m5011/ArenRuss/ERA5_TRAINING_DATA/HURR_LOC_DAILY_GLOBAL.npy')
    env_data = np.load('/global/cfs/cdirs/m5011/ArenRuss/ERA5_TRAINING_DATA/NORMtraining_data_9VAR_1980-2023.npy')
    
    env_data = np.transpose(env_data, (1, 2, 3, 0))
    min_days = min(hurr_loc.shape[0], env_data.shape[0])
    hurr_loc = hurr_loc[:min_days]
    env_data = env_data[:min_days]
    
    region_env = env_data[:, 45:85, 80:180, :]   
    
    if hurr_loc.shape[1] == 90:
        region_hurr = hurr_loc[:, 22:42, 40:90]
    else:
        region_hurr = hurr_loc[:, 45:85, 80:180]
    
    np.save(hurr_path, region_hurr)
    np.save(env_path,  region_env)
    
    print('  region_hurr.shape =', region_hurr.shape)
    print('  region_env.shape  =', region_env.shape)

  region_hurr.shape = (16071, 40, 100)
  region_env.shape  = (16071, 40, 100, 9)


In [4]:
# dfinition of the datasets
import numpy as np
import torch
from torch.utils.data import Dataset

SEQ_LEN = 14
STRIDE = 2
PATCH_H, PATCH_W = 20, 20
N_LAT_PATCHES, N_LON_PATCHES = 2, 5
P = N_LAT_PATCHES * N_LON_PATCHES

class HurricanePatchDataset(Dataset):
    def __init__(self, env_path, hurr_path,
                 seq_len=SEQ_LEN, stride=STRIDE):
        env_all  = np.nan_to_num(np.load(env_path),   nan=0.0, posinf=0.0, neginf=0.0)
        hurr_all = np.nan_to_num(np.load(hurr_path),  nan=0.0, posinf=0.0, neginf=0.0)
        # env_all: (D,40,100,C), hurr_all: (D,40,100)
        self.env      = torch.from_numpy(env_all).float()
        self.hurr     = torch.from_numpy(hurr_all).float()
        self.seq_len  = seq_len

        D = env_all.shape[0]
        # all possible window starts
        all_starts = list(range(0, D - seq_len + 1, stride))
        # keep only windows where at least one frame has a hurricane
        self.starts = [
            s for s in all_starts
            if (hurr_all[s:s+seq_len]          # shape (T,40,100)
                .reshape(seq_len, -1)         # (T, 4000)
                .sum(axis=1) > 0              # (T,)
               ).any()
        ]

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        # 1) which day this window starts at (global index)
        start = self.starts[idx]

        # 2) env sequence: (T,40,100,C) -> (T,C,40,100)
        x = self.env[start:start+self.seq_len]            # (T,40,100,C)
        x = x.permute(0,3,1,2)                             # (T,C,40,100)

        # 3) raw mask sequence: (T,40,100)
        h = self.hurr[start:start+self.seq_len]           # (T,40,100)

        # 4) split into patches: (T, 2,5,20,20)
        y_patch = (
            h.reshape(self.seq_len,
                      N_LAT_PATCHES, PATCH_H,
                      N_LON_PATCHES, PATCH_W)
             .permute(0,1,3,2,4)
        )
        # y_patch: Tensor (T, n_lat_patches, n_lon_patches, PATCH_H, PATCH_W)

        # 5) aggregate to patch‐level: (T, P)
        y_flat = y_patch.reshape(self.seq_len, P, PATCH_H*PATCH_W)
        y      = (y_flat.sum(dim=2) > 0).long()            # (T, P)
        y      = y.view(self.seq_len, N_LAT_PATCHES, N_LON_PATCHES)

        # return: env, patch‐level labels, pixel‐level masks, plus window start idx
        return x, y, y_patch, start

In [5]:
## Timesformer model for cls
import torch
import torch.nn as nn

class FactorizedSTBlock(nn.Module):
    def __init__(self, E, num_heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.norm_sp = nn.LayerNorm(E)
        self.attn_sp = nn.MultiheadAttention(E, num_heads, dropout=drop)
        self.norm_tp = nn.LayerNorm(E)
        self.attn_tp = nn.MultiheadAttention(E, num_heads, dropout=drop)
        self.norm_mlp = nn.LayerNorm(E)
        hid = int(E * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(E, hid),
            nn.GELU(),
            nn.Linear(hid, E),
        )

    def forward(self, x):
        # x: (B, T, P, E)
        B, T, P, E = x.shape

        sp = x.reshape(B*T, P, E).transpose(0,1)
        sp_out, _ = self.attn_sp(sp, sp, sp)
        sp_out = sp_out.transpose(0,1).reshape(B, T, P, E)
        x = x + self.norm_sp(sp_out)

        tp = x.permute(2,0,1,3).reshape(P*B, T, E).transpose(0,1)
        tp_out, _ = self.attn_tp(tp, tp, tp)
        tp_out = tp_out.transpose(0,1).reshape(P, B, T, E).permute(1,2,0,3)
        x = x + self.norm_tp(tp_out)

        # MLP
        mlp_out = self.mlp(self.norm_mlp(x))
        x = x + mlp_out
        return x

class PatchTimesformer(nn.Module):
    def __init__(self,
                 seq_len=7,
                 in_ch=28,
                 patch_h=20, patch_w=20,
                 n_lat=2, n_lon=5,
                 embed_dim=128,
                 depth=4,
                 num_heads=8):
        super().__init__()
        self.seq_len = seq_len
        self.n_lat   = n_lat
        self.n_lon   = n_lon
        self.P       = n_lat * n_lon
        self.patch_dim = in_ch * patch_h * patch_w


        self.patch_proj = nn.Linear(self.patch_dim, embed_dim)
        self.pos_sp = nn.Parameter(torch.zeros(1,1,self.P,embed_dim))
        self.pos_tp = nn.Parameter(torch.zeros(1,seq_len,1,embed_dim))

        # Divided Space–Time Attention
        self.blocks = nn.ModuleList([
            FactorizedSTBlock(embed_dim, num_heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, 1)

        nn.init.trunc_normal_(self.pos_sp, std=0.02)
        nn.init.trunc_normal_(self.pos_tp, std=0.02)
        nn.init.trunc_normal_(self.patch_proj.weight, std=0.02)
        nn.init.zeros_(self.patch_proj.bias)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        # x: (B, T, C, H, W) = (B,7,27,40,100)
        B, T, C, H, W = x.shape

        x = x.view(B, T, C,
                   self.n_lat, H//self.n_lat,
                   self.n_lon, W//self.n_lon)
        x = x.permute(0,1,3,5,2,4,6).contiguous()  # (B,T,2,5,C,20,20)
        x = x.view(B, T, self.P, C, H//self.n_lat, W//self.n_lon)

        x = x.view(B, T, self.P, self.patch_dim)

        x = self.patch_proj(x)

        x = x + self.pos_sp + self.pos_tp

        # ST Attention
        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)  # (B,T,P,E)

        logits = self.head(x).squeeze(-1)

        # reshape → (B, T, 2, 5)
        logits = logits.view(B, T, self.n_lat, self.n_lon)
        return logits


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = PatchTimesformer().to(device)
x = torch.randn(8, 7, 28, 40, 100, device=device)
print("Logits shape:", model(x).shape)

Logits shape: torch.Size([8, 7, 2, 5])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 75
LR     = 1e-4
WD     = 1e-5
best_val_f1 = 0
SEQ_LEN        = 14
STRIDE         = 3
PATCH_H, PATCH_W   = 20, 20
N_LAT_PATCHES      = 2
N_LON_PATCHES      = 5
P = N_LAT_PATCHES * N_LON_PATCHES
IN_CHANNELS = 11  

full_dataset = HurricanePatchDataset('region_env.npy', 'region_hurr.npy', seq_len=SEQ_LEN, stride=STRIDE)

n_train = int(len(full_dataset) * 0.9)
n_val = len(full_dataset) - n_train
train_ds, val_ds = random_split(full_dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=4)

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

def add_pos_embedding(batch_x):
    B, T, C, H, W = batch_x.shape
    lat = torch.linspace(-1, 1, H, device=batch_x.device)
    lon = torch.linspace(-1, 1, W, device=batch_x.device)
    mesh_lat, mesh_lon = torch.meshgrid(lat, lon, indexing='ij')
    # (2, 40, 100) -> (B, T, 2, 40, 100)
    pos = torch.stack([mesh_lat, mesh_lon], dim=0).unsqueeze(0).unsqueeze(0).expand(B, T, -1, -1, -1)
    return torch.cat([batch_x, pos], dim=2)

model = PatchTimesformer(
    seq_len=SEQ_LEN,     
    in_ch=IN_CHANNELS,   
    patch_h=PATCH_H,
    patch_w=PATCH_W,
    n_lat=N_LAT_PATCHES,
    n_lon=N_LON_PATCHES,
    embed_dim=128,
    depth=4,
    num_heads=8
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0.0
    all_train_preds = []
    all_train_targets = []

    for x, y, _, _ in train_loader:
        x, y = x.to(device), y.to(device).float()
        
        if x.shape[2] == 9:
            x = add_pos_embedding(x)

        logits = model(x)                     # (B, T, n_lat, n_lon)
        loss   = criterion(logits, y)         

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).long().cpu().view(-1).numpy()
        targ  = y.cpu().view(-1).numpy()

        all_train_preds.extend(preds)
        all_train_targets.extend(targ)

    avg_train_loss = train_loss / len(train_ds)
    train_acc  = accuracy_score(all_train_targets, all_train_preds)
    train_f1   = f1_score(all_train_targets, all_train_preds, zero_division=0)

    model.eval()
    val_loss = 0.0
    all_val_preds = []
    all_val_targets = []

    with torch.no_grad():
        for x, y, _, _ in val_loader:
            x, y = x.to(device), y.to(device).float()
            
            if x.shape[2] == 9:
                x = add_pos_embedding(x)
                
            logits = model(x)
            val_loss += criterion(logits, y).item() * x.size(0)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long().cpu().view(-1).numpy()
            targ  = y.cpu().view(-1).numpy()

            all_val_preds.extend(preds)
            all_val_targets.extend(targ)

    avg_val_loss = val_loss / len(val_ds)
    val_acc  = accuracy_score(all_val_targets, all_val_preds)
    val_prec = precision_score(all_val_targets, all_val_preds, zero_division=0)
    val_rec  = recall_score(all_val_targets, all_val_preds, zero_division=0)
    val_f1   = f1_score(all_val_targets, all_val_preds, zero_division=0)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_patch_timesformer_by_f1.pt")
        print(f"→ Epoch {epoch:02d}: saved new best model (Val F1: {val_f1:.4f})")

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {avg_train_loss:.4f}, F1: {train_f1:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}"
    )

    if epoch % 5 == 0 or epoch == EPOCHS:
        cm = confusion_matrix(all_val_targets, all_val_preds, labels=[0,1])
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Hurr", "Hurr"])
        fig, ax = plt.subplots(figsize=(4,4))
        disp.plot(ax=ax, cmap='Blues', values_format='d')
        ax.set_title(f"Val Confusion Matrix (Epoch {epoch})")
        plt.show()